## Synthetic

In [14]:
import random
random.seed(0)

In [15]:
import os
import json
import csv
import random
import shutil

random.seed(0)

# Define noise functions
def insert_spaces(text, intensity):
    if not text:
        return text
    n = len(text)
    num_insertions = max(1, int(intensity * n))
    positions = random.sample(range(n + 1), num_insertions)
    positions.sort(reverse=True)
    text_list = list(text)
    for pos in positions:
        text_list.insert(pos, ' ')
    return ''.join(text_list)

def insert_punctuation(text, intensity):
    if not text:
        return text
    n = len(text)
    num_insertions = max(1, int(intensity * n))
    positions = random.sample(range(n + 1), num_insertions)
    positions.sort(reverse=True)
    text_list = list(text)
    punct = ['.', ',', '!', '?', ';', ':']
    for pos in positions:
        p = random.choice(punct)
        text_list.insert(pos, p)
    return ''.join(text_list)

def aeda(sentence, intensity=0.3):
    punctuations = ['.', ',', '!', '?', ';', ':']
    words = sentence.split()
    n_insertions = max(1, int(len(words) * intensity))

    for _ in range(n_insertions):
        insert_punc = random.choice(punctuations)
        insert_pos = random.randint(0, len(words))
        words.insert(insert_pos, insert_punc)

    return ' '.join(words)

def substitute_syllables(text, intensity):
    if len(text) < 2:
        return text
    n = len(text)
    num_changes = max(1, int(intensity * n))
    text_list = list(text)
    for _ in range(num_changes):
        pos = random.randint(0, n - 2)
        new_chars = random.choices('abcdefghijklmnopqrstuvwxyz', k=2)
        text_list[pos] = new_chars[0]
        text_list[pos + 1] = new_chars[1]
    return ''.join(text_list)

# Configure paths
raw_path = os.path.join('dataset', 'raw')
output_base = os.path.join('modified_dataset')
os.makedirs(output_base, exist_ok=True)

noise_config = {
    'space': insert_spaces,
    'punctuation': aeda,
    'syllable': substitute_syllables
}
intensities = [0.1, 0.3, 0.5]

# Process datasets
for noise_type, noise_func in noise_config.items():
    for intensity in intensities:
        # Create noise-intensity directory
        dir_name = f"{noise_type}_{int(intensity * 100)}"
        dest_path = os.path.join(output_base, dir_name)
        shutil.copytree(raw_path, dest_path)
        
        # Modify GSM dataset (test.jsonl)
        gsm_path = os.path.join(dest_path, 'gsm_dataset', 'test.jsonl')
        with open(gsm_path, 'r', encoding='utf-8') as f:
            lines = [json.loads(line) for line in f]
        
        for item in lines:
            item['question'] = noise_func(item['question'], intensity)
            # Preserve answer after "####" to avoid corrupting the final value
            # if '####' in item['answer']:
            #     prefix, _, ans = item['answer'].rpartition('####')
            #     item['answer'] = noise_func(prefix, intensity) + '####' + ans
            # else:
            #     item['answer'] = noise_func(item['answer'], intensity)
        
        with open(gsm_path, 'w', encoding='utf-8') as f:
            for item in lines:
                f.write(json.dumps(item) + '\n')
        
        # Modify Math dataset (math_subset_20.json)
        math_path = os.path.join(dest_path, 'math_dataset', 'math_subset_20.json')
        with open(math_path, 'r', encoding='utf-8') as f:
            math_data = json.load(f)
        
        for key in math_data:
            for subject in math_data[key]:
                for problem in math_data[key][subject]:
                    problem['problem'] = noise_func(problem['problem'], intensity)
                    # problem['solution'] = noise_func(problem['solution'], intensity)
        
        with open(math_path, 'w', encoding='utf-8') as f:
            json.dump(math_data, f, indent=2)
        
        # Modify MMLU dataset (all CSV files)
        mmlu_path = os.path.join(dest_path, 'mmlu_dataset')
        for file in os.listdir(mmlu_path):
            if file.endswith('.csv'):
                file_path = os.path.join(mmlu_path, file)
                rows = []
                with open(file_path, 'r', encoding='utf-8') as f:
                    reader = csv.reader(f)
                    for row in reader:
                        # Modify question
                        modified_row = [noise_func(row[0], intensity)] + [cell for cell in row[1:]]
                        rows.append(modified_row)
                
                with open(file_path, 'w', encoding='utf-8', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerows(rows)
        
        # Modify MultiArith dataset (test.jsonl)
        multiarith_path = os.path.join(dest_path, 'multiarith_dataset', 'test.json')
        with open(multiarith_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        for item in data:
            item['question'] = noise_func(item['question'], intensity)

        with open(multiarith_path, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

print("Dataset modification complete!")

Dataset modification complete!


## R2ATA


In [ ]:
import os
import json

def convert_folder_to_jsonl(input_folder, output_folder):
    # Create output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Iterate through all files in the input folder
    for filename in os.listdir(input_folder):
        file_path = os.path.join(input_folder, filename)

        # Skip directories, only process files
        if os.path.isfile(file_path):
            try:
                # Read the content of the file
                with open(file_path, 'r', encoding='utf-8') as file:
                    content = file.read()

                # Prepare JSON object
                json_obj = {"text": content}

                # Build output file path with .jsonl extension
                base_name = os.path.splitext(filename)[0]
                output_file_path = os.path.join(output_folder, f"{base_name}.jsonl")

                # Write the JSON object to the output file (append mode)
                with open(output_file_path, 'w', encoding='utf-8') as jsonl_file:
                    jsonl_file.write(json.dumps(json_obj, ensure_ascii=False) + "\n")

                print(f"✅ Processed: {filename} -> {output_file_path}")

            except Exception as e:
                print(f"❌ Error processing {filename}: {e}")

# Example usage
if __name__ == "__main__":
    input_folder = "./R2-ATA"
    output_folder = "./R2-ATA-jsonl/"

    convert_folder_to_jsonl(input_folder, output_folder)

## Wikitypo


In [16]:
import pickle

# Load the English typo data
with open('wiki_typos_en_p160000_l5186.pkl', 'rb') as f:
    en_typos = pickle.load(f)

# Load the Hindi typo data
with open('wiki_typos_en_p320000_l9368.pkl', 'rb') as f:
    hi_typos = pickle.load(f)

# Create dictionaries
en_typo_dict = {typo: correct for typo, correct in en_typos}
en_2_typo_dict = {typo: correct for typo, correct in hi_typos}

# Combine both dictionaries (if needed)
combined_typo_dict = {**en_typo_dict, **en_2_typo_dict}

# Now you have:
# en_typo_dict - English typos to corrections
# hi_typo_dict - Hindi typos to corrections
# combined_typo_dict - Combined dictionary of both

In [17]:
import os
import json
import csv
from pathlib import Path


def process_text(text):
    words = text.split()
    processed_words = []
    for word in words:
        # Check if word is in typo dict (case sensitive)
        if word in combined_typo_dict:
            processed_words.append(combined_typo_dict[word])
        else:
            processed_words.append(word)
    return ' '.join(processed_words)

def process_jsonl_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8') as outfile:
        for line in infile:
            data = json.loads(line)
            if 'question' in data:
                data['question'] = process_text(data['question'])
            if 'problem' in data:
                data['problem'] = process_text(data['problem'])
            outfile.write(json.dumps(data) + '\n')

def process_csv_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8', newline='') as outfile:
        reader = csv.reader(infile)
        writer = csv.writer(outfile)
        for row in reader:
            processed_row = [process_text(cell) for cell in row]
            writer.writerow(processed_row)

def process_json_file(input_path, output_path):
    with open(input_path, 'r', encoding='utf-8') as infile:
        data = json.load(infile)
    
    # Process JSON data recursively
    def process_dict(d):
        for key, value in d.items():
            if isinstance(value, str):
                if key == "problem":
                    d[key] = process_text(value)
            elif isinstance(value, dict):
                process_dict(value)
            elif isinstance(value, list):
                for i, item in enumerate(value):
                    if isinstance(item, str):
                        value[i] = process_text(item)
                    elif isinstance(item, dict):
                        process_dict(item)
    
    process_dict(data)
    
    with open(output_path, 'w', encoding='utf-8') as outfile:
        json.dump(data, outfile, ensure_ascii=False, indent=2)

def process_ma_file(input_path, output_path):           
    with open(input_path, 'r', encoding='utf-8') as infile, \
         open(output_path, 'w', encoding='utf-8') as outfile:
        data = json.load(infile)
        for questions in data:
            if 'question' in data:
                data['question'] = process_text(data['question'])
        
        json.dump(data, outfile, ensure_ascii=False, indent=2)

def process_directory(input_dir, output_dir):
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    for root, dirs, files in os.walk(input_dir):
        # Create corresponding subdirectories in output directory
        rel_path = os.path.relpath(root, input_dir)
        current_output_dir = os.path.join(output_dir, rel_path)
        os.makedirs(current_output_dir, exist_ok=True)
        
        for file in files:
            input_path = os.path.join(root, file)
            file_name, file_ext = os.path.splitext(file)
            output_path = os.path.join(current_output_dir, f"{file_name}{file_ext}")
            
            try:
                if file_ext == '.jsonl':
                    process_jsonl_file(input_path, output_path)
                elif file_ext == '.csv':
                    process_csv_file(input_path, output_path)
                elif file_ext == '.json':
                    if "multiarith" in output_path:
                        process_ma_file(input_path, output_path)
                    else:
                        process_json_file(input_path, output_path)
                else:
                    # For other file types, copy as-is
                    with open(input_path, 'rb') as infile, \
                         open(output_path, 'wb') as outfile:
                        outfile.write(infile.read())
                print(f"Processed: {input_path} -> {output_path}")
            except Exception as e:
                print(f"Error processing {input_path}: {str(e)}")

# Main execution
if __name__ == "__main__":
    base_dir = '.'  # Current directory where your datasets are
    data_dir = 'dataset/raw/'  
    output_dir = os.path.join(base_dir, 'wikitypo')
    
    # Process each dataset directory
    datasets = ['gsm_dataset', 'math_dataset', 'mmlu_dataset', 'multiarith_dataset']
    for dataset in datasets:
        input_dataset_dir = os.path.join(data_dir, dataset)
        print(input_dataset_dir)
        if os.path.exists(input_dataset_dir):
            output_dataset_dir = os.path.join(output_dir, dataset)
            process_directory(input_dataset_dir, output_dataset_dir)

dataset/raw/gsm_dataset
Processed: dataset/raw/gsm_dataset\test.jsonl -> .\wikitypo\gsm_dataset\.\test.jsonl
dataset/raw/math_dataset
Processed: dataset/raw/math_dataset\math_subset_20.json -> .\wikitypo\math_dataset\.\math_subset_20.json
dataset/raw/mmlu_dataset
Processed: dataset/raw/mmlu_dataset\abstract_algebra_test.csv -> .\wikitypo\mmlu_dataset\.\abstract_algebra_test.csv
Processed: dataset/raw/mmlu_dataset\anatomy_test.csv -> .\wikitypo\mmlu_dataset\.\anatomy_test.csv
Processed: dataset/raw/mmlu_dataset\astronomy_test.csv -> .\wikitypo\mmlu_dataset\.\astronomy_test.csv
Processed: dataset/raw/mmlu_dataset\business_ethics_test.csv -> .\wikitypo\mmlu_dataset\.\business_ethics_test.csv
Processed: dataset/raw/mmlu_dataset\clinical_knowledge_test.csv -> .\wikitypo\mmlu_dataset\.\clinical_knowledge_test.csv
Processed: dataset/raw/mmlu_dataset\college_biology_test.csv -> .\wikitypo\mmlu_dataset\.\college_biology_test.csv
Processed: dataset/raw/mmlu_dataset\college_chemistry_test.csv -> 